# GLiNER2 Synthetic Data Generator Demo

This notebook demonstrates the `DataGenerator` class from `generate.py`, which turns a natural language task description into synthetic GLiNER2-style training examples.

It also includes a scaffold for fine-tuning `fastino/gliner2-base-v1` on the generated data using the GLiNER2 training API.

In [ ]:
from pprint import pprint

from generate import DataGenerator

generator = DataGenerator(seed=42)

## Single-task examples

We first generate examples for each of the four GLiNER2 task types:

- NER
- Classification
- Relation extraction
- JSON extraction


In [ ]:
single_task_descriptions = [
    "Extract company, person and location names from short tech news snippets.",  # NER
    "Classify customer review sentiment into positive, negative and neutral.",    # Classification
    "Identify works_for and lives_in relations between people and organizations.", # Relation extraction
    "Extract product name, storage and price into a JSON structure.",             # JSON extraction
]

for desc in single_task_descriptions:
    print("==== Description ====")
    print(desc)
    examples = generator.generate(desc, n=3)
    print("First example:")
    pprint(examples[0])
    print()

## Multi-task examples

Now we use composite task descriptions that implicitly require multiple task types to be active at once.


In [ ]:
multi_task_descriptions = [
    # NER + classification
    "From each customer review, extract company names and classify overall sentiment into positive, negative or neutral.",
    # JSON extraction + classification
    "Extract a JSON product record (name, storage, price) from each sentence and classify whether the tone is positive, negative or neutral.",
]

for desc in multi_task_descriptions:
    print("==== Multi-task description ====")
    print(desc)
    examples = generator.generate(desc, n=4)
    print("Example 0:")
    pprint(examples[0])
    print("Example 1:")
    pprint(examples[1])
    print()

## Bonus: Fine-tune GLiNER2 on generated data (scaffold)

The following cells show a minimal workflow for fine-tuning `fastino/gliner2-base-v1` on a synthetic sentiment classification task using the generated JSONL data.

This is **scaffold code**: you may need to adjust batch sizes, epochs, and device configuration depending on your hardware.


In [ ]:
import json
from pathlib import Path

from generate import DataGenerator

data_dir = Path("synthetic_data")
data_dir.mkdir(exist_ok=True)

sentiment_description = (
    "Classify customer review sentiment into positive, negative and neutral. "
    "Make the examples diverse across topics."
)

train_gen = DataGenerator(seed=123)
eval_gen = DataGenerator(seed=456)

train_examples = train_gen.generate(sentiment_description, n=80)
eval_examples = eval_gen.generate(sentiment_description, n=40)

train_path = data_dir / "train.jsonl"
eval_path = data_dir / "eval.jsonl"

with train_path.open("w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with eval_path.open("w", encoding="utf-8") as f:
    for ex in eval_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_path, eval_path

In [ ]:
# Fine-tuning scaffold. This follows the GLiNER2 training docs.
# You may need a GPU and to install the `gliner2` package.

from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

model_name = "fastino/gliner2-base-v1"
model = GLiNER2.from_pretrained(model_name)

config = TrainingConfig(
    output_dir="./output_gliner2_sentiment",
    experiment_name="gliner2_sentiment_demo",
    num_epochs=3,
    batch_size=8,
    encoder_lr=5e-5,
    task_lr=5e-5,
)

trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=str(train_path))

In [ ]:
# Simple evaluation scaffold on the held-out synthetic set.
# We compare the base GLiNER2 model vs the fine-tuned model.
import json

# Load a fresh base model (un-fine-tuned) for comparison
from gliner2 import GLiNER2   # pyright: ignore[reportMissingImports]
model_name = "fastino/gliner2-base-v1"
base_model = GLiNER2.from_pretrained(model_name)

base_correct = 0
ft_correct = 0
num_total = 0

with eval_path.open("r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        text = ex["input"]
        true_cls = ex["output"]["classifications"][0]["true_label"][0]

        # Base model prediction
        base_pred = base_model.classify_text(
            text,
            tasks={"sentiment": {"labels": ["positive", "negative", "neutral"]}},
            threshold=0.5,
            format_results=True,
            include_confidence=False,
        )
        base_cls = base_pred.get("sentiment")

        # Fine-tuned model prediction
        ft_pred = model.classify_text(
            text,
            tasks={"sentiment": {"labels": ["positive", "negative", "neutral"]}},
            threshold=0.5,
            format_results=True,
            include_confidence=False,
        )
        ft_cls = ft_pred.get("sentiment")

        if base_cls == true_cls:
            base_correct += 1
        if ft_cls == true_cls:
            ft_correct += 1
        num_total += 1

results = {
    "base_accuracy": base_correct / max(1, num_total),
    "fine_tuned_accuracy": ft_correct / max(1, num_total),
}
results